# HalluGuard × MiniCheck — full benchmark

Runs all 29,320 LLM-AggreFact test rows through `MiniCheck.score()`. HalluGuard uses deterministic no-thinking exact `YES`/`NO`; failures are recorded without guessed labels.


## 1. Setup


In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORKING = Path("/kaggle/working")
if not WORKING.exists():
    WORKING = Path.cwd()
SOURCE_DIR = WORKING / "minicheck_halluguard_binary_source"
MINICHECK_COMMIT = "b58b9fa69acbd1015ec970fa65dd752413a053d2"

def run_checked(args, cwd=None):
    print("+", " ".join(map(str, args)))
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True, text=True)

gpu_before = subprocess.run(
    ["nvidia-smi"], text=True, capture_output=True, check=True
).stdout
print(gpu_before)
assert "T4" in gpu_before, "This notebook targets Kaggle T4 GPUs."

if SOURCE_DIR.exists():
    shutil.rmtree(SOURCE_DIR)
run_checked([
    "git", "clone", "--filter=blob:none", "https://github.com/Liyan06/MiniCheck.git", str(SOURCE_DIR)
])
run_checked(["git", "checkout", MINICHECK_COMMIT], cwd=SOURCE_DIR)

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "vllm==0.10.2", "transformers==4.57.6", "datasets", "nltk", "scikit-learn",
    "pandas", "huggingface_hub", "accelerate"
])

# Read-only credential handling. Never place token values in notebook source or output.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN is available from the environment/Kaggle Secrets (value not printed).")
else:
    print("HF_TOKEN is not set; continuing because the referenced public assets may not require it.")


Fri Sep 25 13:47:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Cloning into '/kaggle/working/minicheck_halluguard_binary_source'...


+ git checkout b58b9fa69acbd1015ec970fa65dd752413a053d2
+ /usr/bin/python3 -m pip install -q vllm==0.10.2 transformers==4.57.6 datasets nltk scikit-learn pandas huggingface_hub accelerate


Note: switching to 'b58b9fa69acbd1015ec970fa65dd752413a053d2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at b58b9fa keep only code changes


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 M

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.7.0 which is incompatible.
gradio 5.50.0 requires starlette<1.0,>=0.40.0, but you have starlette 1.7.0 which is incompatible.


HF_TOKEN is available from the environment/Kaggle Secrets (value not printed).


## 2. MiniCheck binary HalluGuard adapter


In [2]:
import ast
from pathlib import Path

mini_path = SOURCE_DIR / "minicheck" / "minicheck.py"
inf_path = SOURCE_DIR / "minicheck" / "inference.py"
mini = mini_path.read_text(encoding="utf-8")
inf = inf_path.read_text(encoding="utf-8")

def replace_exact(text, old, new, label, expected=1):
    found = text.count(old)
    if found != expected:
        raise RuntimeError(f"{label}: expected {expected} anchor(s), found {found}")
    return text.replace(old, new, expected)

allowed_tail = "'Granite-Guardian-3.3-8B']"
mini = replace_exact(
    mini,
    allowed_tail,
    "'Granite-Guardian-3.3-8B', 'HalluGuard-Qwen3-4B']",
    "MiniCheck allowed-model list",
    expected=2,
)

anchor = """        elif model_name == 'Granite-Guardian-3.3-8B':
            if not max_tokens or max_tokens<2048:
                print(\"For Granite Guardian 3.3 - fixing the max_tokens to be 2048\")
                max_tokens=2048

            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )"""
halluguard_constructor = """        elif model_name == 'HalluGuard-Qwen3-4B':
            if not max_tokens or max_tokens < 2:
                raise ValueError(\"HalluGuard max_tokens must be at least 2.\")
            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )
"""
mini = replace_exact(
    mini, anchor, halluguard_constructor + anchor,
    "MiniCheck HalluGuard constructor",
)

model_anchor = "        if model_id == 'Bespoke-MiniCheck-7B':\n"
inf = replace_exact(
    inf,
    model_anchor,
    "        self.model_revision = None\n        self.vllm_kwargs = {}\n\n" + model_anchor,
    "LLMCheck defaults",
)

anchor = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        else:
            raise ValueError(\"model_id must be 'Bespoke-MiniCheck-7B'\")"""
replacement = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        elif model_id == 'HalluGuard-Qwen3-4B':
            self.model_id = 'lrsbrgrn/HalluGuard-Qwen3-4B'
            self.operating_mode = \"halluguard_binary\"
            self.model_revision = '2bf3858f8bd0c22b0c410f6b0c855098727a2554'
            self.vllm_kwargs = {
                'enforce_eager': True,
                'disable_custom_all_reduce': True,
                'gpu_memory_utilization': 0.80,
                'max_num_seqs': 4,
            }
        else:
            raise ValueError(\"Unsupported LLMCheck model_id: %s\" % model_id)"""
inf = replace_exact(inf, anchor, replacement, "LLMCheck model dispatch")

inf = replace_exact(
    inf,
    "            model=self.model_id, ",
    "            model=self.model_id,\n            revision=self.model_revision, ",
    "vLLM revision",
)
inf = replace_exact(
    inf,
    "            enable_prefix_caching=self.enable_prefix_caching",
    "            enable_prefix_caching=self.enable_prefix_caching,\n            **self.vllm_kwargs",
    "vLLM HalluGuard stability kwargs",
)

anchor = """        self.sampling_params = SamplingParams(
            temperature=0,
            max_tokens=self.max_tokens,
            stop_token_ids=terminators,
            logprobs=5
        )"""
replacement = """        if self.operating_mode == \"halluguard_binary\":
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=[self.tokenizer.eos_token_id],
                seed=42,
            )
        else:
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=terminators,
                logprobs=5
            )
        self.last_halluguard_records = []"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard sampling parameters")

anchor = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        return text"""
replacement = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        elif self.operating_mode == \"halluguard_binary\":
            prompt = (
                \"Document:\\n\" + doc + \"\\n\\nClaim:\\n\" + claim
                + \"\\n\\nReply exactly YES if the claim is fully supported by the document. \"
                + \"Reply exactly NO if the claim is contradicted or unsupported by the document. \"
                + \"No explanation, reasoning, punctuation, XML, or extra text.\"
            )
            text = self.tokenizer.apply_chat_template(
                [{\"role\": \"user\", \"content\": prompt}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        return text"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard chat template")

inf = replace_exact(
    inf,
    "class LLMCheck:\n",
    "class HalluGuardParseError(ValueError):\n    pass\n\n\nclass LLMCheck:\n",
    "HalluGuard parse exception",
)

parser_anchor = '    def get_support_prob_hybrid_gg(self, response, marker="score"):\n'
parser = """    def parse_halluguard_output(self, response_text):
        final_text = response_text.strip()
        if final_text == \"YES\":
            return \"GROUNDED\", 1.0, final_text
        if final_text == \"NO\":
            return \"HALLUCINATED\", 0.0, final_text
        raise HalluGuardParseError(
            \"HalluGuard binary output must be exactly trimmed uppercase YES or NO. \"
            + \"Raw output: \" + repr(response_text[:4000])
        )

    def get_support_score_halluguard(self, response):
        raw_text = response.outputs[0].text
        label, score, final_text = self.parse_halluguard_output(raw_text)
        self.last_halluguard_records.append({
            \"classification\": label,
            \"support_score\": score,
            \"raw_output\": raw_text,
        })
        return score

"""
inf = replace_exact(
    inf, parser_anchor, parser + parser_anchor,
    "HalluGuard strict parser",
)

anchor = """        if self.operating_mode==\"bespoke\":
            probs_per_chunk_sentence = [self.get_support_prob(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode==\"gg_hybrid\":
            probs_per_chunk_sentence = [self.get_support_prob_hybrid_gg(responses[idx]) for idx in range(len(responses))]"""
replacement = anchor + """
        elif self.operating_mode == \"halluguard_binary\":
            self.last_halluguard_records = []
            probs_per_chunk_sentence = [
                self.get_support_score_halluguard(response) for response in responses
            ]"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard response dispatch")

ast.parse(mini, filename=str(mini_path))
ast.parse(inf, filename=str(inf_path))
mini_path.write_text(mini, encoding="utf-8")
inf_path.write_text(inf, encoding="utf-8")
print("Patched pinned MiniCheck with deterministic HalluGuard binary support.")

run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(SOURCE_DIR)])

import nltk
for resource in ("punkt", "punkt_tab"):
    nltk.download(resource, quiet=True)

sys.path.insert(0, str(SOURCE_DIR))
from minicheck.minicheck import MiniCheck
from minicheck.inference import HalluGuardParseError
print("MiniCheck import path:", __import__("minicheck").__file__)


Patched pinned MiniCheck with deterministic HalluGuard binary support.
+ /usr/bin/python3 -m pip install -q -e /kaggle/working/minicheck_halluguard_binary_source
MiniCheck import path: /kaggle/working/minicheck_halluguard_binary_source/minicheck/__init__.py


In [3]:
import json
import platform
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

MODEL_NAME = "HalluGuard-Qwen3-4B"
MODEL_ID = "lrsbrgrn/HalluGuard-Qwen3-4B"
MODEL_REVISION = "2bf3858f8bd0c22b0c410f6b0c855098727a2554"
DATASET_ID = "lytang/LLM-AggreFact"
SPLIT = "test"
CHUNK_SIZE = 500
MAX_MODEL_LEN = 4096
MAX_TOKENS = 8
TENSOR_PARALLEL_SIZE = 1
ENABLE_PREFIX_CACHING = False
RANDOM_STATE = 42
CACHE_DIR = str(WORKING / "hf_cache")

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

OUTPUT_DIR = WORKING / "halluguard_minicheck_full_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


['Tesla T4']


## 3. Data and model


In [4]:
raw = load_dataset(DATASET_ID, split=SPLIT, token=hf_token)
eval_df = raw.to_pandas()
eval_df["source_index"] = np.arange(len(eval_df), dtype=np.int64)
eval_df = eval_df.sort_values(["dataset", "source_index"], kind="stable").reset_index(drop=True)
dataset_names = sorted(eval_df["dataset"].unique())
assert len(dataset_names) == 11 and len(eval_df) == 29320

PILOT_PER_DATASET = 5
SUB_BATCH_SIZE = 4
CHECKPOINT_PATH = OUTPUT_DIR / "predictions_checkpoint.csv"
FINAL_PATH = OUTPUT_DIR / "predictions_final.csv"
METRICS_PATH = OUTPUT_DIR / "dataset_metrics.csv"
RESUME_SOURCE = None

# Kaggle notebook outputs are read-only inputs in a new session. Recover only
# successful rows; prior failures must be retried and are archived for debugging.
input_root = Path("/kaggle/input")
if not CHECKPOINT_PATH.exists() and input_root.exists():
    candidates = sorted(input_root.rglob("predictions_checkpoint.csv"))
    inspected = []
    for candidate in candidates:
        try:
            frame = pd.read_csv(candidate)
            required = {"source_index", "dataset", "label", "status"}
            if required.issubset(frame.columns):
                inspected.append((int((frame["status"] == "OK").sum()), len(frame), candidate, frame))
        except Exception as exc:
            print(f"Ignoring unreadable checkpoint {candidate}: {type(exc).__name__}: {exc}")
    if inspected:
        _, _, selected, imported = max(inspected, key=lambda item: (item[0], item[1], str(item[2])))
        shutil.copy2(selected, OUTPUT_DIR / "imported_checkpoint_original.csv")
        recovered = imported.loc[imported["status"] == "OK"].copy()
        valid_keys = eval_df[["source_index", "dataset", "label"]]
        recovered = recovered.merge(valid_keys, on=["source_index", "dataset", "label"], how="inner")
        sort_col = "attempt_count" if "attempt_count" in recovered.columns else "source_index"
        recovered = recovered.sort_values(sort_col).drop_duplicates("source_index", keep="last")
        recovered.to_csv(CHECKPOINT_PATH, index=False)
        RESUME_SOURCE = str(selected)
        print(f"Recovered {len(recovered)} successful rows from {selected}; failed rows will be retried.")

print(f"Loaded {len(eval_df):,} test rows across {len(dataset_names)} datasets.")


README.md: 0.00B [00:00, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/35.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/30420 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/29320 [00:00<?, ? examples/s]

Loaded 29,320 test rows across 11 datasets.


In [5]:
scorer = MiniCheck(
    model_name="HalluGuard-Qwen3-4B",
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    max_tokens=MAX_TOKENS,
    enable_prefix_caching=ENABLE_PREFIX_CACHING,
    cache_dir=CACHE_DIR,
)
assert scorer.model.operating_mode == "halluguard_binary"
assert scorer.model.model_id == MODEL_ID
assert scorer.model.model_revision == MODEL_REVISION
assert scorer.model.vllm_kwargs == {
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "gpu_memory_utilization": 0.80,
    "max_num_seqs": 4,
}
print("Loaded HalluGuard through MiniCheck on", TENSOR_PARALLEL_SIZE, "GPUs.")


INFO 09-25 13:51:28 [__init__.py:216] Automatically detected platform cuda.
INFO 09-25 13:51:30 [utils.py:328] non-default args: {'download_dir': '/kaggle/working/hf_cache', 'dtype': torch.float16, 'seed': 2024, 'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.8, 'max_num_seqs': 4, 'disable_log_stats': True, 'revision': '2bf3858f8bd0c22b0c410f6b0c855098727a2554', 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': 'lrsbrgrn/HalluGuard-Qwen3-4B'}


config.json: 0.00B [00:00, ?B/s]

INFO 09-25 13:51:48 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 09-25 13:51:48 [__init__.py:2767] Casting torch.bfloat16 to torch.float16.
INFO 09-25 13:51:48 [__init__.py:1815] Using max model len 4096
INFO 09-25 13:51:55 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-25 13:51:55 [__init__.py:3400] Cudagraph is disabled under eager mode


...

INFO 09-25 13:52:07 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=176) INFO 09-25 13:52:09 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=176) INFO 09-25 13:52:09 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='lrsbrgrn/HalluGuard-Qwen3-4B', speculative_config=None, tokenizer='lrsbrgrn/HalluGuard-Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=2bf3858f8bd0c22b0c410f6b0c855098727a2554, tokenizer_revision=2bf3858f8bd0c22b0c410f6b0c855098727a2554, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir='/kaggle/working/hf_cache', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reas

[W925 13:52:13.041889061 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W925 13:52:13.042823390 socket.cpp:200] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W925 13:52:13.043262670 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=176) INFO 09-25 13:52:13 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=176) WARNING 09-25 13:52:13 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
(EngineCore_DP0 pid=176) INFO 09-25 13:52:13 [gpu_model_runner.py:2338

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:23<00:23, 23.66s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:32<00:00, 15.05s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:32<00:00, 16.34s/it]
(EngineCore_DP0 pid=176) 


(EngineCore_DP0 pid=176) INFO 09-25 13:53:12 [default_loader.py:268] Loading weights took 32.81 seconds
(EngineCore_DP0 pid=176) INFO 09-25 13:53:13 [gpu_model_runner.py:2392] Model loading took 7.5023 GiB and 57.887664 seconds
(EngineCore_DP0 pid=176) INFO 09-25 13:53:17 [gpu_worker.py:298] Available KV cache memory: 3.53 GiB
(EngineCore_DP0 pid=176) INFO 09-25 13:53:17 [kv_cache_utils.py:864] GPU KV cache size: 25,664 tokens
(EngineCore_DP0 pid=176) INFO 09-25 13:53:17 [kv_cache_utils.py:868] Maximum concurrency for 4,096 tokens per request: 6.27x
(EngineCore_DP0 pid=176) INFO 09-25 13:53:17 [gpu_worker.py:391] Free memory on device (14.44/14.56 GiB) on startup. Desired GPU memory utilization is (0.8, 11.65 GiB). Actual usage is 7.5 GiB for weight, 0.61 GiB for peak activation, 0.01 GiB for non-torch memory, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3628948889` to fit into requested memory, or `--kv-cache-memory=6629409792` to ful

## 4. Checkpointed pilot then full run


In [6]:
RESULT_COLUMNS = [
    "source_index", "dataset", "label", "prediction", "support_score", "status",
    "error_type", "error_message", "raw_model_output", "elapsed_seconds", "attempt_count",
]

def empty_results():
    return pd.DataFrame(columns=RESULT_COLUMNS)

def load_results():
    if not CHECKPOINT_PATH.exists():
        return empty_results()
    frame = pd.read_csv(CHECKPOINT_PATH)
    required = {"source_index", "dataset", "label", "prediction", "support_score", "status"}
    missing_required = required - set(frame.columns)
    if missing_required:
        raise ValueError(f"Checkpoint schema mismatch; missing columns: {sorted(missing_required)}")
    optional_defaults = {
        "error_type": None, "error_message": None, "raw_model_output": None,
        "elapsed_seconds": None, "attempt_count": 1,
    }
    for column, default in optional_defaults.items():
        if column not in frame.columns:
            frame[column] = default
    frame = frame[frame["source_index"].isin(eval_df["source_index"])].copy()
    frame["attempt_count"] = frame["attempt_count"].fillna(1).astype(int)
    frame = frame[RESULT_COLUMNS]
    return frame.sort_values("attempt_count").drop_duplicates("source_index", keep="last")

def save_results(frame):
    ordered = frame.sort_values(["dataset", "source_index"], kind="stable").reset_index(drop=True)
    tmp = CHECKPOINT_PATH.with_suffix(".csv.tmp")
    ordered.to_csv(tmp, index=False)
    os.replace(tmp, CHECKPOINT_PATH)

def exception_chain(exc):
    seen = set()
    while exc is not None and id(exc) not in seen:
        seen.add(id(exc))
        yield exc
        exc = exc.__cause__ or exc.__context__

def is_parse_failure(exc):
    return any(isinstance(item, HalluGuardParseError) for item in exception_chain(exc))

def is_engine_dead(exc):
    for item in exception_chain(exc):
        name = type(item).__name__.lower()
        message = str(item).lower()
        if "enginedead" in name or "engine core" in message and "dead" in message:
            return True
    return False

def one_record(row, prior_attempt):
    started = time.perf_counter()
    try:
        preds, probs, _, _ = scorer.score(
            docs=[str(row.doc)], claims=[str(row.claim)], chunk_size=CHUNK_SIZE
        )
        return {
            "source_index": int(row.source_index), "dataset": row.dataset,
            "label": int(row.label), "prediction": int(preds[0]),
            "support_score": float(probs[0]), "status": "OK",
            "error_type": None, "error_message": None, "raw_model_output": None,
            "elapsed_seconds": time.perf_counter() - started,
            "attempt_count": prior_attempt + 1,
        }, False
    except Exception as exc:
        parse_failure = is_parse_failure(exc)
        message = str(exc)[:8000]
        return {
            "source_index": int(row.source_index), "dataset": row.dataset,
            "label": int(row.label), "prediction": None, "support_score": None,
            "status": "PARSE_FAILURE" if parse_failure else "INFERENCE_FAILURE",
            "error_type": type(exc).__name__, "error_message": message,
            "raw_model_output": message if parse_failure else None,
            "elapsed_seconds": time.perf_counter() - started,
            "attempt_count": prior_attempt + 1,
        }, is_engine_dead(exc)

def counts(frame):
    return {
        "attempted": len(frame),
        "successful": int((frame["status"] == "OK").sum()),
        "parse_failures": int((frame["status"] == "PARSE_FAILURE").sum()),
        "inference_failures": int((frame["status"] == "INFERENCE_FAILURE").sum()),
    }

def upsert(current, records):
    if not records:
        return current
    combined = pd.concat([current, pd.DataFrame(records, columns=RESULT_COLUMNS)], ignore_index=True)
    return combined.sort_values("attempt_count").drop_duplicates("source_index", keep="last")

results = load_results()
done = set(results.loc[results["status"] == "OK", "source_index"].astype(int))
prior = dict(zip(results["source_index"].astype(int), results["attempt_count"].astype(int)))
pilot = (
    eval_df.groupby("dataset", group_keys=False)
    .sample(n=PILOT_PER_DATASET, random_state=RANDOM_STATE)
    .sort_values(["dataset", "source_index"], kind="stable")
)
phases = [("pilot", pilot), ("full", eval_df)]
ENGINE_ABORTED = False
ENGINE_ABORT_RECORD = None

for phase_name, frame in phases:
    buffered = []
    phase_indices = set(frame["source_index"].astype(int))
    for row in frame.itertuples(index=False):
        source_index = int(row.source_index)
        if source_index in done:
            continue
        record, fatal_engine = one_record(row, prior.get(source_index, 0))
        buffered.append(record)
        prior[source_index] = int(record["attempt_count"])
        if record["status"] == "OK":
            done.add(source_index)

        if len(buffered) >= SUB_BATCH_SIZE or fatal_engine:
            results = upsert(results, buffered)
            save_results(results)
            buffered = []

        if fatal_engine:
            ENGINE_ABORTED = True
            ENGINE_ABORT_RECORD = record
            (OUTPUT_DIR / "engine_abort.json").write_text(
                json.dumps(record, indent=2, default=str), encoding="utf-8"
            )
            print("vLLM engine died; checkpoint saved and remaining rows were NOT falsely marked failed.")
            break

    if buffered:
        results = upsert(results, buffered)
        save_results(results)

    phase_result = results[results["source_index"].isin(phase_indices)]
    print(f"{phase_name} counts:", counts(phase_result))
    if ENGINE_ABORTED:
        break

print("Actual checkpoint counts:", counts(results))


Tokenizing: 100%|██████████| 1/1 [00:00<00:00, 19.67it/s]


Adding requests:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore_DP0 pid=176) WARNING 09-25 13:53:37 [cudagraph_dispatcher.py:102] cudagraph dispatching keys are not initialized. No cudagraph will be used.


Tokenizing: 100%|██████████| 1/1 [00:00<00:00, 129.73it/s]


...

Tokenizing: 100%|██████████| 1/1 [00:00<00:00, 43.27it/s]


Adding requests:   0%|          | 0/7 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/7 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore_DP0 pid=176) ERROR 09-25 17:10:36 [dump_input.py:69] Dumping input data for V1 LLM engine (v0.10.2) with config: model='lrsbrgrn/HalluGuard-Qwen3-4B', speculative_config=None, tokenizer='lrsbrgrn/HalluGuard-Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=2bf3858f8bd0c22b0c410f6b0c855098727a2554, tokenizer_revision=2bf3858f8bd0c22b0c410f6b0c855098727a2554, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir='/kaggle/working/hf_cache', load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None), seed=

(EngineCore_DP0 pid=176) Process EngineCore_DP0:
(EngineCore_DP0 pid=176) Traceback (most recent call last):
(EngineCore_DP0 pid=176)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore_DP0 pid=176)     self.run()
(EngineCore_DP0 pid=176)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore_DP0 pid=176)     self._target(*self._args, **self._kwargs)
(EngineCore_DP0 pid=176)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 722, in run_engine_core
(EngineCore_DP0 pid=176)     raise e
(EngineCore_DP0 pid=176)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 711, in run_engine_core
(EngineCore_DP0 pid=176)     engine_core.run_busy_loop()
(EngineCore_DP0 pid=176)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 738, in run_busy_loop
(EngineCore_DP0 pid=176)     self._process_engine_step()
(EngineCore_DP0 pid=176)   File "/usr/local/l

## 5. Metrics and artifacts

Metrics use successful rows only; parse and inference failures are reported separately.


In [7]:
results = load_results()
merged = eval_df.merge(
    results, on=["source_index", "dataset", "label"], how="left", validate="one_to_one"
)
merged["status"] = merged["status"].fillna("NOT_RUN")

metric_rows = []
for dataset_name, group in merged.groupby("dataset", sort=True):
    valid = group[group["status"] == "OK"]
    row_metrics = {"accuracy": None, "f1": None, "balanced_accuracy": None}
    if len(valid):
        y_true = valid["label"].astype(int)
        y_pred = valid["prediction"].astype(int)
        row_metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
        row_metrics["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
        if y_true.nunique() == 2:
            row_metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    metric_rows.append({
        "dataset": dataset_name,
        "total_samples": len(group),
        "successful_samples": len(valid),
        "coverage": len(valid) / len(group),
        "parse_failures": int((group["status"] == "PARSE_FAILURE").sum()),
        "inference_failures": int((group["status"] == "INFERENCE_FAILURE").sum()),
        "not_run": int((group["status"] == "NOT_RUN").sum()),
        "parse_failure_rate": int((group["status"] == "PARSE_FAILURE").sum()) / len(group),
        **row_metrics,
    })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(METRICS_PATH, index=False)
merged.to_csv(FINAL_PATH, index=False)
merged[merged["status"] == "PARSE_FAILURE"].to_csv(
    OUTPUT_DIR / "parse_failures_raw_output.csv", index=False
)

failures = merged[merged["status"].isin(["PARSE_FAILURE", "INFERENCE_FAILURE"])]
error_summary = (
    failures.groupby(["status", "error_type", "error_message"], dropna=False)
    .size().reset_index(name="count").sort_values("count", ascending=False)
)
error_summary.to_csv(OUTPUT_DIR / "failure_error_summary.csv", index=False)

valid_all = merged[merged["status"] == "OK"]
overall = {
    "total_samples": len(merged),
    "successful_samples": len(valid_all),
    "parse_failures": int((merged["status"] == "PARSE_FAILURE").sum()),
    "inference_failures": int((merged["status"] == "INFERENCE_FAILURE").sum()),
    "not_run": int((merged["status"] == "NOT_RUN").sum()),
    "parse_failure_rate": int((merged["status"] == "PARSE_FAILURE").sum()) / len(merged),
    "accuracy_successful_rows_only": None,
    "f1_successful_rows_only": None,
    "balanced_accuracy_successful_rows_only": None,
    "macro_balanced_accuracy_successful_rows_only": (
        float(metrics["balanced_accuracy"].mean())
        if metrics["balanced_accuracy"].notna().any() else None
    ),
}
if len(valid_all):
    y_true = valid_all["label"].astype(int)
    y_pred = valid_all["prediction"].astype(int)
    overall["accuracy_successful_rows_only"] = float(accuracy_score(y_true, y_pred))
    overall["f1_successful_rows_only"] = float(f1_score(y_true, y_pred, zero_division=0))
    if y_true.nunique() == 2:
        overall["balanced_accuracy_successful_rows_only"] = float(
            balanced_accuracy_score(y_true, y_pred)
        )

try:
    import vllm
    vllm_version = vllm.__version__
except Exception:
    vllm_version = None

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": DATASET_ID,
    "split": SPLIT,
    "rows": len(eval_df),
    "chunk_size": CHUNK_SIZE,
    "max_model_len": MAX_MODEL_LEN,
    "max_tokens": MAX_TOKENS,
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "enable_prefix_caching": ENABLE_PREFIX_CACHING,
    "generation": {"enable_thinking": False, "temperature": 0, "format": "exact YES/NO"},
    "metric_handling": "Accuracy, F1, and balanced accuracy exclude failed/not-run rows; coverage and failure counts are reported explicitly.",
    "resume_source": RESUME_SOURCE,
    "engine_aborted": ENGINE_ABORTED,
    "counts_and_metrics": overall,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "vllm": vllm_version,
    "cuda": torch.version.cuda,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str), encoding="utf-8"
)
display(metrics)
display(error_summary.head(20))
print("Actual full benchmark counts and metrics:", json.dumps(overall, indent=2))
if ENGINE_ABORTED:
    print("Run stopped safely after vLLM engine death; remaining rows are NOT_RUN and can be resumed.")
elif overall["not_run"]:
    print("Run is incomplete; rerun with the checkpoint to continue.")
else:
    print("All rows were attempted. Actual parse/inference failure counts are shown above.")


,dataset,total_samples,successful_samples,coverage,parse_failures,inference_failures,not_run,parse_failure_rate,accuracy,f1,balanced_accuracy
0,AggreFact-CNN,558,558,1.000000,0,0,0,0.0,0.876344,0.932683,0.573537
1,AggreFact-XSum,558,558,1.000000,0,0,0,0.0,0.639785,0.719665,0.633950
2,ClaimVerify,1088,1088,1.000000,0,0,0,0.0,0.765625,0.855688,0.607850
3,ExpertQA,3702,557,0.150459,0,1,3144,0.0,0.763016,0.857451,0.570900
4,FactCheck-GPT,1566,5,0.003193,0,0,1561,0.0,0.400000,0.000000,NaN
5,Lfqa,1911,5,0.002616,0,0,1906,0.0,0.600000,0.666667,0.583333
6,RAGTruth,16371,5,0.000305,0,0,16366,0.0,1.000000,1.000000,NaN
7,Reveal,1710,5,0.002924,0,0,1705,0.0,1.000000,1.000000,1.000000
8,TofuEval-MediaS,726,5,0.006887,0,0,721,0.0,0.800000,0.857143,0.750000
9,TofuEval-MeetB,772,5,0.006477,0,0,767,0.0,0.600000,0.750000,0.375000


,status,error_type,error_message,count
0,INFERENCE_FAILURE,EngineDeadError,EngineCore encountered an issue. See stack tra...,1


Actual full benchmark counts and metrics: {
  "total_samples": 29320,
  "successful_samples": 2796,
  "parse_failures": 0,
  "inference_failures": 1,
  "not_run": 26523,
  "parse_failure_rate": 0.0,
  "accuracy_successful_rows_only": 0.7621602288984263,
  "f1_successful_rows_only": 0.8515293592319714,
  "balanced_accuracy_successful_rows_only": 0.6158389846229955,
  "macro_balanced_accuracy_successful_rows_only": 0.677174533378705
}
Run stopped safely after vLLM engine death; remaining rows are NOT_RUN and can be resumed.
